In [2]:
import numpy as np
import pandas as pd

data = (
    pd.read_csv("atp_matches_2008.csv")
      .drop_duplicates()
      .sort_values("tourney_date")
)

data = data[["surface", "winner_name", "loser_name"]].dropna()

player_elo = {}

K_multiplier = 40

# print(len(data))
# print(data["surface"].unique())

predictions = []
losses = []

def new_player():
    return {
        "Hard": [1200],
        "Grass": [1200],
        "Clay": [1200],
        "Carpet": [120],
    }

for row in data.itertuples(index=False):
    if row.winner_name not in player_elo:
        player_elo[row.winner_name] = new_player()
    if row.loser_name not in player_elo:
        player_elo[row.loser_name] = new_player()

for row in data.itertuples(index=False):
    court_type = row.surface
    winner_rating = player_elo[row.winner_name][court_type][-1]
    loser_rating = player_elo[row.loser_name][court_type][-1]

    expected_a = (1 / (1 + (10 ** (((loser_rating) - (winner_rating)) / 400))))   
    expected_b = 1 - expected_a

    
    if expected_a >= 0.5:
        predictions.append(1)
    else:
        predictions.append(0)

    loss = -np.log(expected_a)
    losses.append(loss)

    # print(expected_a + expected_b)

    result_a = winner_rating + (K_multiplier * (1 - expected_a))
    result_b = loser_rating + (K_multiplier * (0 - expected_b))

    # print(result_a, result_b)

    player_elo[row.winner_name][court_type].append((result_a))
    player_elo[row.loser_name][court_type].append((result_b))

print(player_elo)





{'Dejan Petrovic': {'Hard': [1200, 1220.0, 1202.1621375799175, 1180.88431458515, 1157.801370477055], 'Grass': [1200, 1183.492893454798, 1203.2262341698226, 1183.3134736893176, 1165.2415326210928], 'Clay': [1200], 'Carpet': [120]}, 'Stephane Huet': {'Hard': [1200, 1180.0, 1161.8064632478, 1140.8635552107653, 1169.9146181490723, 1151.0419510238446, 1133.1138513754272, 1125.8332395918733, 1110.981916745766, 1097.6764393795277], 'Grass': [1200, 1178.8499774443092, 1164.5099041574101, 1186.5458021228892, 1168.31873519258, 1189.3689533453478], 'Clay': [1200, 1220.0, 1202.2412958569114, 1182.8460115996336, 1168.3612346583402, 1146.7459098392346, 1130.0931660576587, 1111.2513975291442, 1136.2518887657793, 1120.7989018280427, 1144.65304802161], 'Carpet': [120, 101.04499364683112, 80.9848391583894, 63.508757439167596]}, 'Magnus Norman': {'Hard': [1200, 1220.0, 1198.8499774443092, 1218.9161778223724, 1238.9785674397383, 1257.8211672808288, 1275.5834165579895, 1291.2439585356205, 1309.432945036085